In [1]:
!pip install -q pandas numpy requests beautifulsoup4 tqdm lxml

In [2]:
import pandas as pd
import numpy as np
import requests
import re
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime
from tqdm import tqdm

In [3]:
OUTPUT_FILE = "fast-food-scrape.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

KATALOG_URLS = [
    {
        "url": "https://menukuliner.net/katalog/fast-food",
        "raw_category": "fast food",
        "city": "unknown",
        "keyword": "fast food"
    },
    {
        "url": "https://menukuliner.net/katalog/ayam-goreng",
        "raw_category": "fried chicken",
        "city": "unknown",
        "keyword": "ayam goreng"
    },
    {
        "url": "https://menukuliner.net/katalog/fried-chicken",
        "raw_category": "fried chicken",
        "city": "unknown",
        "keyword": "fried chicken"
    },
    {
        "url": "https://menukuliner.net/katalog/burger",
        "raw_category": "burger",
        "city": "unknown",
        "keyword": "burger"
    },
    {
        "url": "https://menukuliner.net/katalog/pizza",
        "raw_category": "pizza",
        "city": "unknown",
        "keyword": "pizza"
    },
    {
        "url": "https://menukuliner.net/katalog/kentang-goreng",
        "raw_category": "fries",
        "city": "unknown",
        "keyword": "kentang goreng"
    },
    {
        "url": "https://menukuliner.net/katalog/hotdog",
        "raw_category": "hotdog",
        "city": "unknown",
        "keyword": "hotdog"
    },
    {
        "url": "https://menukuliner.net/katalog/sandwich",
        "raw_category": "sandwich",
        "city": "unknown",
        "keyword": "sandwich"
    },
    {
        "url": "https://menukuliner.net/katalog/kfc",
        "raw_category": "fast food chain",
        "city": "unknown",
        "keyword": "kfc"
    },
    {
        "url": "https://menukuliner.net/katalog/mcdonalds",
        "raw_category": "fast food chain",
        "city": "unknown",
        "keyword": "mcdonalds"
    },
    {
        "url": "https://menukuliner.net/katalog/burger-king",
        "raw_category": "fast food chain",
        "city": "unknown",
        "keyword": "burger king"
    },
    {
        "url": "https://menukuliner.net/katalog/richeese-factory",
        "raw_category": "fast food chain",
        "city": "unknown",
        "keyword": "richeese factory"
    },
    {
        "url": "https://menukuliner.net/katalog/pizza-hut",
        "raw_category": "fast food chain",
        "city": "unknown",
        "keyword": "pizza hut"
    },
]

MAX_LINKS = 100
REQUEST_DELAY = 1

In [4]:
def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_price(text):
    if text is None or pd.isna(text):
        return np.nan

    text = str(text)
    text = re.sub(r"[^0-9]", "", text)

    if text == "":
        return np.nan

    return int(text)


def is_price(text):
    text = clean_text(text)
    return bool(re.search(r"Rp\s*[0-9][0-9\.\,]*", text))


def is_valid_menu_name(text):
    text = clean_text(text)
    lower = text.lower()

    if len(text) < 3:
        return False

    if len(text) > 120:
        return False

    if lower.startswith("rp"):
        return False

    if re.fullmatch(r"[0-9\.\,\s]+", text):
        return False

    noise_words = [
        "nama menu harga",
        "nama menu",
        "harga",
        "harga menu",
        "daftar harga",
        "delivery",
        "gofood",
        "gojek",
        "grabfood",
        "shopeefood",
        "menukuliner",
        "restaurant",
        "restoran",
        "promo",
        "diskon",
        "review",
        "rating",
        "alamat",
        "jam buka",
        "telepon",
        "cukup merogoh",
        "dibanderol",
        "disajikan",
        "berkisar",
        "pilihan menu",
        "siapkan uang",
        "tidak mahal",
        "untuk menyantap",
        "harga yang dibanderol",
        "anda bisa",
        "di bawah ini",
        "berikut ini",
        "terbaru",
        "halaman",
        "lihat",
        "baca juga",
    ]

    if any(word in lower for word in noise_words):
        return False

    return True


def infer_raw_category(text):
    text = str(text).lower()

    if any(k in text for k in ["kfc", "richeese", "mcd", "mcdonald", "burger king", "pizza hut", "phd"]):
        return "fast food chain"

    if any(k in text for k in ["fried chicken", "ayam goreng", "fire chicken", "chicken wings", "wing", "ayam crispy", "crispy chicken"]):
        return "fried chicken"

    if any(k in text for k in ["burger", "cheeseburger", "beef burger", "chicken burger", "big mac", "whopper"]):
        return "burger"

    if any(k in text for k in ["pizza", "stuffed crust", "personal pizza", "pan pizza"]):
        return "pizza"

    if any(k in text for k in ["kentang", "fries", "french fries", "potato wedges", "wedges"]):
        return "fries"

    if any(k in text for k in ["hotdog", "hot dog"]):
        return "hotdog"

    if any(k in text for k in ["sandwich", "subway"]):
        return "sandwich"

    if any(k in text for k in ["combo", "paket", "box", "meal", "set", "hemat"]):
        return "combo meal"

    if any(k in text for k in ["nugget", "nuggets", "chicken strip", "chicken sticks"]):
        return "fast food snack"

    return "fast food"


def infer_city_from_text(text):
    text = str(text).lower()

    cities = [
        "jakarta", "semarang", "bandung", "yogyakarta", "surabaya",
        "medan", "tangerang", "bekasi", "bogor", "depok",
        "malang", "solo", "denpasar", "balikpapan", "makassar"
    ]

    for city in cities:
        if city in text:
            return city.title()

    return "Unknown"

In [5]:
def collect_menu_links():
    collected = []

    for katalog in tqdm(KATALOG_URLS, desc="Collecting menu links"):
        url = katalog["url"]

        try:
            response = requests.get(url, headers=HEADERS, timeout=30)

            if response.status_code != 200:
                print(f"Skip {url} | status {response.status_code}")
                continue

            soup = BeautifulSoup(response.text, "lxml")

            for a in soup.find_all("a", href=True):
                href = a["href"]

                if "/menu/" not in href:
                    continue

                full_url = urljoin("https://menukuliner.net", href)

                if "menukuliner.net/menu/" not in full_url:
                    continue

                collected.append({
                    "source_platform": "MenuKuliner",
                    "source_url": full_url,
                    "raw_category_from_katalog": katalog["raw_category"],
                    "keyword": katalog["keyword"],
                    "city_from_katalog": katalog["city"]
                })

        except Exception as e:
            print(f"Failed katalog: {url} | {e}")

        time.sleep(REQUEST_DELAY)

    df_links = pd.DataFrame(collected)

    if df_links.empty:
        return df_links

    df_links = df_links.drop_duplicates(subset=["source_url"]).reset_index(drop=True)
    df_links = df_links.head(MAX_LINKS)

    return df_links


df_links = collect_menu_links()

print(df_links.shape)
display(df_links.head(20))

Skip https://menukuliner.net/katalog/fast-food | status 404


Skip https://menukuliner.net/katalog/burger | status 404


Skip https://menukuliner.net/katalog/pizza | status 404


Skip https://menukuliner.net/katalog/kentang-goreng | status 404


Skip https://menukuliner.net/katalog/hotdog | status 404


...

Skip https://menukuliner.net/katalog/kfc | status 404


Skip https://menukuliner.net/katalog/mcdonalds | status 404


Skip https://menukuliner.net/katalog/richeese-factory | status 404


Skip https://menukuliner.net/katalog/pizza-hut | status 404
(100, 5)


,source_platform,source_url,raw_category_from_katalog,keyword,city_from_katalog
0,MenuKuliner,https://menukuliner.net/menu/6754/mie-pangsit-...,fried chicken,ayam goreng,unknown
1,MenuKuliner,https://menukuliner.net/menu/22036/pawon-kreme...,fried chicken,ayam goreng,unknown
2,MenuKuliner,https://menukuliner.net/menu/24870/sakana-cafe,fried chicken,ayam goreng,unknown
3,MenuKuliner,https://menukuliner.net/menu/42714/warung-mba-...,fried chicken,ayam goreng,unknown
4,MenuKuliner,https://menukuliner.net/menu/67119/ayam-dan-le...,fried chicken,ayam goreng,unknown
5,MenuKuliner,https://menukuliner.net/menu/95485/kedai-manna...,fried chicken,ayam goreng,unknown
6,MenuKuliner,https://menukuliner.net/menu/135428/warung-mam...,fried chicken,ayam goreng,unknown
7,MenuKuliner,https://menukuliner.net/menu/192554/sate-ayam-...,fried chicken,ayam goreng,unknown
8,MenuKuliner,https://menukuliner.net/menu/247147/bakso-benh...,fried chicken,ayam goreng,unknown
9,MenuKuliner,https://menukuliner.net/menu/275664/cafe-thirt...,fried chicken,ayam goreng,unknown


In [6]:
def get_restaurant_name(soup):
    h1 = soup.find("h1")

    if h1:
        title = clean_text(h1.get_text())
    else:
        title_tag = soup.find("title")
        title = clean_text(title_tag.get_text()) if title_tag else "Unknown Restaurant"

    title = title.replace("Daftar Harga Menu Delivery", "")
    title = title.replace("Daftar Harga Menu", "")
    title = title.replace("Terbaru", "")
    title = re.sub(r"\s+", " ", title)
    title = title.strip(" ,-")

    if title == "":
        return "Unknown Restaurant"

    return title


def extract_menu_from_table(soup):
    items = []

    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            cells = [clean_text(td.get_text()) for td in tr.find_all(["td", "th"])]

            if len(cells) < 2:
                continue

            name_candidate = cells[0]
            price_candidate = cells[-1]

            if not is_price(price_candidate):
                continue

            if not is_valid_menu_name(name_candidate):
                continue

            items.append({
                "section": None,
                "menu_name": name_candidate,
                "price": clean_price(price_candidate),
                "extract_method": "html_table"
            })

    return items


def extract_menu_from_text(soup):
    text = soup.get_text("\n")
    lines = [clean_text(line) for line in text.splitlines()]
    lines = [line for line in lines if line]

    items = []
    current_section = None

    for idx, line in enumerate(lines):
        if line.lower().startswith("harga menu"):
            current_section = clean_text(line.replace("Harga Menu", ""))
            continue

        if not is_price(line):
            continue

        price = clean_price(line)

        if pd.isna(price):
            continue

        candidates = []

        if idx - 1 >= 0:
            candidates.append(lines[idx - 1])

        if idx - 2 >= 0:
            candidates.append(lines[idx - 2])

        menu_name = None

        for candidate in candidates:
            if is_valid_menu_name(candidate):
                menu_name = candidate
                break

        if menu_name is None:
            continue

        items.append({
            "section": current_section,
            "menu_name": menu_name,
            "price": price,
            "extract_method": "text_pattern"
        })

    return items


def deduplicate_menu_items(items):
    unique = []
    seen = set()

    for item in items:
        if pd.isna(item["price"]):
            continue

        key = (
            str(item.get("section")).lower(),
            item["menu_name"].lower(),
            int(item["price"])
        )

        if key not in seen:
            seen.add(key)
            unique.append(item)

    return unique


def scrape_menu_page(target):
    rows = []
    url = target["source_url"]

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")

        restaurant_name = get_restaurant_name(soup)

        menu_items = []
        menu_items.extend(extract_menu_from_table(soup))
        menu_items.extend(extract_menu_from_text(soup))
        menu_items = deduplicate_menu_items(menu_items)

        city = infer_city_from_text(restaurant_name + " " + url)

        if city == "Unknown" and target["city_from_katalog"] != "unknown":
            city = str(target["city_from_katalog"]).title()

        for item in menu_items:
            raw_category = infer_raw_category(
                str(target["raw_category_from_katalog"]) + " " +
                str(target["keyword"]) + " " +
                str(item["section"]) + " " +
                str(item["menu_name"]) + " " +
                str(restaurant_name)
            )

            rows.append({
                "restaurant_name": restaurant_name,
                "city": city,
                "raw_category": raw_category,
                "section": item["section"],
                "menu_name": item["menu_name"],
                "price": item["price"],
                "source_platform": target["source_platform"],
                "source_url": url,
                "search_keyword": target["keyword"],
                "extract_method": item["extract_method"],
                "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })

    except Exception as e:
        print(f"Failed: {url} | {e}")

    return rows

In [7]:
all_rows = []

for _, target in tqdm(df_links.iterrows(), total=len(df_links), desc="Scraping menu pages"):
    rows = scrape_menu_page(target)
    all_rows.extend(rows)

    if all_rows:
        pd.DataFrame(all_rows).to_csv("checkpoint_fast_food_scrape.csv", index=False)

    time.sleep(REQUEST_DELAY)

df_fast_food = pd.DataFrame(all_rows)

if df_fast_food.empty:
    print("Tidak ada data berhasil discrape.")
else:
    df_fast_food = df_fast_food.drop_duplicates(
        subset=["restaurant_name", "menu_name", "price", "source_url"],
        keep="first"
    ).reset_index(drop=True)

    df_fast_food["price"] = pd.to_numeric(df_fast_food["price"], errors="coerce")

    df_fast_food = df_fast_food[
        (df_fast_food["price"].isna()) |
        ((df_fast_food["price"] >= 3000) & (df_fast_food["price"] <= 500000))
    ].copy()

    df_fast_food = df_fast_food.reset_index(drop=True)

    df_fast_food.to_csv(OUTPUT_FILE, index=False)

    print("=" * 80)
    print("SCRAPING FAST FOOD SELESAI")
    print("=" * 80)
    print(f"Output file        : {OUTPUT_FILE}")
    print(f"Total rows         : {len(df_fast_food)}")
    print(f"Unique restaurants : {df_fast_food['restaurant_name'].nunique()}")
    print(f"Unique cities      : {df_fast_food['city'].nunique()}")
    print(f"Source platform    : {df_fast_food['source_platform'].unique().tolist()}")
    print(f"Min price          : {df_fast_food['price'].min()}")
    print(f"Median price       : {df_fast_food['price'].median()}")
    print(f"Max price          : {df_fast_food['price'].max()}")

    display(df_fast_food.head(30))

Scraping menu pages: 100%|██████████| 100/100 [02:55<00:00,  1.76s/it]

SCRAPING FAST FOOD SELESAI
Output file        : fast-food-scrape.csv
Total rows         : 8044
Unique restaurants : 100
Unique cities      : 11
Source platform    : ['MenuKuliner']
Min price          : 3000
Median price       : 24000.0
Max price          : 357500


,restaurant_name,city,raw_category,section,menu_name,price,source_platform,source_url,search_keyword,extract_method,scraped_at
0,"Mie Pangsit CW, Di Depan Kings Tattoo Ubud, Ba...",Unknown,fried chicken,None,Pangsit Ayam 5 Pcs,16500,MenuKuliner,https://menukuliner.net/menu/6754/mie-pangsit-...,ayam goreng,html_table,2026-04-28 03:14:32
1,"Mie Pangsit CW, Di Depan Kings Tattoo Ubud, Ba...",Unknown,fried chicken,None,Gyoza 5 Pcs,38000,MenuKuliner,https://menukuliner.net/menu/6754/mie-pangsit-...,ayam goreng,html_table,2026-04-28 03:14:32
2,"Mie Pangsit CW, Di Depan Kings Tattoo Ubud, Ba...",Unknown,fried chicken,None,Mie Pangsit Babi Reguler Mie Babi Dengan Pangs...,26000,MenuKuliner,https://menukuliner.net/menu/6754/mie-pangsit-...,ayam goreng,html_table,2026-04-28 03:14:32
3,"Mie Pangsit CW, Di Depan Kings Tattoo Ubud, Ba...",Unknown,fried chicken,None,Mie babi komplit Mie Babi Dengan Pangsit Ayam ...,32000,MenuKuliner,https://menukuliner.net/menu/6754/mie-pangsit-...,ayam goreng,html_table,2026-04-28 03:14:32
4,"Mie Pangsit CW, Di Depan Kings Tattoo Ubud, Ba...",Unknown,fried chicken,None,Mie Pangsit Babi Jumbo Mie Babi Dengan Porsi B...,36000,MenuKuliner,https://menukuliner.net/menu/6754/mie-pangsit-...,ayam goreng,html_table,2026-04-28 03:14:32
5,"Mie Pangsit CW, Di Depan Kings Tattoo Ubud, Ba...",Unknown,fried chicken,None,Mie Pangsit Ayam Reguler Mie Ayam Dengan Pangs...,22000,MenuKuliner,https://menukuliner.net/menu/6754/mie-pangsit-...,ayam goreng,html_table,2026-04-28 03:14:32
6,"Mie Pangsit CW, Di Depan Kings Tattoo Ubud, Ba...",Unknown,fried chicken,None,Mie Pangsit Ayam Jumbo Mie Ayam Porsi Besar,28000,MenuKuliner,https://menukuliner.net/menu/6754/mie-pangsit-...,ayam goreng,html_table,2026-04-28 03:14:32
7,"Mie Pangsit CW, Di Depan Kings Tattoo Ubud, Ba...",Unknown,fried chicken,None,Mie Ayam Bakso Mie Ayam Dengan Pangsit Dan Bak...,25000,MenuKuliner,https://menukuliner.net/menu/6754/mie-pangsit-...,ayam goreng,html_table,2026-04-28 03:14:32
8,"Mie Pangsit CW, Di Depan Kings Tattoo Ubud, Ba...",Unknown,fried chicken,None,Mie Babi Bakso Mie Babi Dengan Pangsit Ayam Da...,30000,MenuKuliner,https://menukuliner.net/menu/6754/mie-pangsit-...,ayam goreng,html_table,2026-04-28 03:14:32
9,"Mie Pangsit CW, Di Depan Kings Tattoo Ubud, Ba...",Unknown,fried chicken,None,Mie Ayam Bakso Jumbo Mie Ayam Dengan Pangsit A...,31000,MenuKuliner,https://menukuliner.net/menu/6754/mie-pangsit-...,ayam goreng,html_table,2026-04-28 03:14:32


In [8]:
print("Shape:", df_fast_food.shape)

display(df_fast_food["raw_category"].value_counts())
display(df_fast_food["restaurant_name"].value_counts().head(20))
display(df_fast_food.sample(min(20, len(df_fast_food)), random_state=42))

Shape: (8044, 11)


,count
raw_category,
fried chicken,5098
fast food chain,2946


,count
restaurant_name,
"Juice Bang Mitun, Gunung Putri, Jakarta 2026",242
"Cuppa Coffee Bistro, Ruko Paramount Rodeo, Jakarta 2026",236
"The Larder at 55, Gandapura, Bandung 2026",213
"Burger King, Antasari, Bandar Lampung 2026",157
"Mulanis Cafe and Resto, Sidokarto, Yogyakarta 2026",156
"Burger King, Raden Inten, Bandar Lampung 2026",156
"Burger King, Cikarang Baru Raya, Jakarta 2026",155
"Burger King, Hayam Wuruk, Jakarta 2026",155
"Burger King, Duren Sawit, Jakarta 2026",155


,restaurant_name,city,raw_category,section,menu_name,price,source_platform,source_url,search_keyword,extract_method,scraped_at
743,"Warteg Pipa, Teluknaga, Jakarta 2026",Jakarta,fried chicken,Sop dan Soto,Nasi + Suiran Ayam + Tomat + Sambal + Emping,25000,MenuKuliner,https://menukuliner.net/menu/581330/warteg-pip...,ayam goreng,text_pattern,2026-04-28 03:14:56
4414,"SAN GYU by Hangry, Senopati, Jakarta 2026",Jakarta,fried chicken,Extras,Seasoned rice with seaweed and sesame,13000,MenuKuliner,https://menukuliner.net/menu/514864/san-gyu-by...,fried chicken,text_pattern,2026-04-28 03:16:19
5813,"Top Express Chicken n Burger, Tilatang Kamang,...",Unknown,fast food chain,None,SPAGETTI ORI,16500,MenuKuliner,https://menukuliner.net/menu/173946/top-expres...,burger king,html_table,2026-04-28 03:16:52
3532,"Casy Mpasi Bayi, Cab Galaxy, Jakarta 2026",Jakarta,fried chicken,None,Blend Cream Soup Hati Ayam,18000,MenuKuliner,https://menukuliner.net/menu/277459/casy-mpasi...,fried chicken,html_table,2026-04-28 03:16:01
3817,"Cuppa Coffee Bistro, Ruko Paramount Rodeo, Jak...",Jakarta,fried chicken,None,Roasted Duck Noodle,75500,MenuKuliner,https://menukuliner.net/menu/288610/cuppa-coff...,fried chicken,html_table,2026-04-28 03:16:08
1755,"Kedai Luckyto, Balikpapan 2026",Balikpapan,fried chicken,None,Sambal Goreng Kentang 125gr,22000,MenuKuliner,https://menukuliner.net/menu/39089/kedai-luckyto,ayam goreng,html_table,2026-04-28 03:15:22
7367,"Burger King, Solo Square, Solo 2026",Solo,fast food chain,King's Chicken [Baru],+ Rp 21.000,34000,MenuKuliner,https://menukuliner.net/menu/900755/burger-kin...,burger king,text_pattern,2026-04-28 03:17:15
3154,"Bevande Cafe, Bulak Timur, Jakarta 2026",Jakarta,fried chicken,None,Fried Chicken Ayam Krispi(Bagian Bawah)+ Saus ...,15000,MenuKuliner,https://menukuliner.net/menu/262370/bevande-ca...,fried chicken,html_table,2026-04-28 03:15:54
3434,"Casy Catering Mpasi Bayi Pondok Gede, Margonda...",Jakarta,fried chicken,None,Hati Ayam Cream Soup,18000,MenuKuliner,https://menukuliner.net/menu/277458/casy-cater...,fried chicken,html_table,2026-04-28 03:15:59
518,"Cahaya Surga Mulut Lamongan, Jakarta 2026",Jakarta,fried chicken,None,Udang Asam Manis,30000,MenuKuliner,https://menukuliner.net/menu/275915/cahaya-sur...,ayam goreng,html_table,2026-04-28 03:14:49


In [10]:
from google.colab import files

files.download("fast-food-scrape.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>